In [1]:

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import IsolationForest
import joblib

In [2]:
df = pd.read_csv('dados/station_data_dataverse.csv')
df.columns = df.columns.str.lower()   
print(df.shape)
print(df.columns.tolist())

(3395, 24)
['sessionid', 'kwhtotal', 'dollars', 'created', 'ended', 'starttime', 'endtime', 'chargetimehrs', 'weekday', 'platform', 'distance', 'userid', 'stationid', 'locationid', 'managervehicle', 'facilitytype', 'mon', 'tues', 'wed', 'thurs', 'fri', 'sat', 'sun', 'reportedzip']


In [3]:
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 3395 entries, 0 to 3394
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sessionid       3395 non-null   int64  
 1   kwhtotal        3395 non-null   float64
 2   dollars         3395 non-null   float64
 3   created         3395 non-null   str    
 4   ended           3395 non-null   str    
 5   starttime       3395 non-null   int64  
 6   endtime         3395 non-null   int64  
 7   chargetimehrs   3395 non-null   float64
 8   weekday         3395 non-null   str    
 9   platform        3395 non-null   str    
 10  distance        2330 non-null   float64
 11  userid          3395 non-null   int64  
 12  stationid       3395 non-null   int64  
 13  locationid      3395 non-null   int64  
 14  managervehicle  3395 non-null   int64  
 15  facilitytype    3395 non-null   int64  
 16  mon             3395 non-null   int64  
 17  tues            3395 non-null   int64  
 18 

,sessionid,kwhtotal,dollars,starttime,endtime,chargetimehrs,distance,userid,stationid,locationid,managervehicle,facilitytype,mon,tues,wed,thurs,fri,sat,sun,reportedzip
count,3.395000e+03,3395.000000,3395.000000,3395.000000,3395.000000,3395.000000,2330.000000,3.395000e+03,3395.000000,3395.000000,3395.000000,3395.000000,3395.000000,3395.000000,3395.000000,3395.000000,3395.000000,3395.000000,3395.000000,3395.000000
mean,5.487001e+06,5.809629,0.118268,13.743446,16.455965,2.841488,18.652378,5.742395e+07,576789.678056,629934.460677,0.595582,2.428571,0.181443,0.187040,0.210015,0.216495,0.179676,0.018262,0.007069,0.703976
std,2.590657e+06,2.892727,0.492562,3.204370,3.406732,1.507472,11.420571,2.674772e+07,257486.310402,255620.993849,0.490851,0.811204,0.385442,0.390001,0.407379,0.411916,0.383974,0.133918,0.083793,0.456569
min,1.004821e+06,0.000000,0.000000,0.000000,0.000000,0.012500,0.856911,1.042767e+07,129465.000000,125372.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,3.234666e+06,4.350000,0.000000,11.000000,14.000000,2.110278,5.135871,3.329548e+07,369001.000000,481066.000000,0.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,5.451498e+06,6.230000,0.000000,13.000000,16.000000,2.808889,21.023826,4.924181e+07,549414.000000,503205.000000,1.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
75%,7.746644e+06,6.830000,0.000000,17.000000,20.000000,3.544167,27.285053,8.188052e+07,864630.000000,878393.000000,1.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
max,9.998981e+06,23.680000,7.500000,23.000000,23.000000,55.238056,43.059292,9.834581e+07,995505.000000,978130.000000,1.000000,4.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [4]:
df.columns = df.columns.str.lower()

colunas_uteis = ['sessionid', 'userid', 'created', 'ended', 'kwhtotal', 'chargetimehrs', 'starttime']
df = df[colunas_uteis].drop_duplicates()

df['created'] = pd.to_datetime(df['created'], errors='coerce')
df['ended'] = pd.to_datetime(df['ended'], errors='coerce')

df = df.dropna(subset=['userid', 'kwhtotal', 'chargetimehrs', 'starttime'])
df = df[(df['kwhtotal'] > 0) & (df['chargetimehrs'] > 0)]
df.shape

(3340, 7)

In [5]:
colunas = ['kwhtotal', 'chargetimehrs', 'starttime']
X = df[colunas]

modelo = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
modelo.fit(X)

df['rotulo'] = modelo.predict(X)
df['score'] = modelo.decision_function(X)
df['rotulo'].value_counts()

rotulo
 1    3173
-1     167
Name: count, dtype: int64

In [6]:
df['media_usuario'] = df.groupby('userid')['kwhtotal'].transform('mean')
df['regra_dobro'] = df['kwhtotal'] > 2 * df['media_usuario']
df['regra_dobro'].sum()

np.int64(16)

In [ ]:
df['anomalia'] = (df['rotulo'] == -1) | df['regra_sobro']

df['motivo'] = ''
df.loc[df['rotulo'] == -1, 'motivo'] += 'padrão atípico (modelo); '
df.loc[df['regra_dobro'], 'motivo'] += 'mais que 2x a média do usuário; '

df['anomalia'].sum()